# FABRIC Service Marketplace

## Step1: Create the slice
Create a slice with 3 nodes, portal, gateway and service node. Portal and gateway have fabnetv4ext. Gateway and service node have fabnetV6.

In [ ]:
from fabrictestbed_extensions.fablib.fablib import FablibManager as fablib_manager
fablib = fablib_manager()
fablib.show_config();

In [ ]:
slice_name = 'service_marketplace'

portal_node_name = 'Node1'
portal_public_nic_name = 'nic1'
portal_public_network_name = 'net1'

gateway_node_name = 'Node2'
gateway_public_nic_name = 'nic2'
gateway_fabnetv6_nic_name = 'nic3'
gateway_public_network_name = 'net2'
gatewway_fabnetv6_network_name = 'net3'

portal_site = "UCSD"
gateway_site = "NCSA"
service_node_site = "STAR"


#Create Slice
slice = fablib.new_slice(name=slice_name)

# portal
portal_node = slice.add_node(name=portal_node_name,site=portal_site, image="docker_ubuntu_22", cores=4, ram=8)
iface1 = portal_node.add_component(model='NIC_Basic', name=portal_public_nic_name).get_interfaces()[0]


# gateway
gateway_node = slice.add_node(name=gateway_node_name,site=gateway_site, image="docker_ubuntu_22", cores=4, ram=8)
iface2 = gateway_node.add_component(model='NIC_Basic', name=gateway_public_nic_name).get_interfaces()[0]
iface3 = gateway_node.add_component(model='NIC_Basic', name=gateway_fabnetv6_nic_name).get_interfaces()[0]

# Service node
service_node_name = 'service_node1'
service_node = slice.add_node(name=service_node_name, site=service_node_site)
service_node.add_fabnet(net_type='IPv6')
    
# Networks
#net1 = slice.add_l3network(name=portal_public_network_name, interfaces=[iface1], type='IPv6')
portal_public_network = slice.add_l3network(name=portal_public_network_name, interfaces=[iface1], type='IPv4Ext')
gateway_public_network = slice.add_l3network(name=gateway_public_network_name, interfaces=[iface2], type='IPv4Ext')
gateway_fabnetv6_network = slice.add_l3network(name=gatewway_fabnetv6_network_name, interfaces=[iface3], type='IPv6')



#Submit Slice Request
slice.submit();


## Step2: Config the node

In [ ]:
slice=fablib.get_slice(name=slice_name)


portal_public_network = slice.get_network(name=portal_public_network_name)
portal_public_network_available_ips = portal_public_network.get_available_ips()
portal_public_network.show()

gateway_public_network = slice.get_network(name=gateway_public_network_name)
gateway_public_network_available_ips =  gateway_public_network.get_available_ips()
gateway_public_network.show()

gatewway_fabnetv6_network = slice.get_network(name=gatewway_fabnetv6_network_name)
gatewway_fabnetv6_network_available_ips =  gatewway_fabnetv6_network.get_available_ips()
gatewway_fabnetv6_network.show()



In [ ]:
#configure gateway fabnet v6
gateway_node = slice.get_node(name=gateway_node_name)        
gateway_fabnet_iface = gateway_node.get_interface(network_name=gatewway_fabnetv6_network_name)  
gateway_fabnet_addr = gatewway_fabnetv6_network_available_ips.pop(0)
gateway_fabnet_iface.ip_addr_add(gateway_fabnet_addr, subnet=gatewway_fabnetv6_network.get_subnet())
gateway_node.ip_route_add(subnet=fablib.FABNETV6_SUBNET, gateway=gatewway_fabnetv6_network.get_gateway())
stdout, stderr = gateway_node.execute(f'ip addr show {gateway_fabnet_iface.get_device_name()}')
stdout, stderr = gateway_node.execute(f'ip route list')
stdout, stderr = gateway_node.execute('ip -6 route list')
    


In [ ]:
#configure fabnetext
try:
    # Enable Public IPv4 make_ip_publicly_routable
    gateway_public_network.make_ip_publicly_routable(ipv4=[str(gateway_public_network_available_ips[0])])
    portal_public_network.make_ip_publicly_routable(ipv4=[str(portal_public_network_available_ips[0])])
    slice.submit()
except Exception as e:
    print(f"Exception: {e}")
    import traceback
    traceback.print_exc()


### Configure routing table

In [ ]:
from ipaddress import IPv4Address, IPv4Network
slice = fablib.get_slice(name=slice_name)
gateway_public_network= slice.get_network(name=gateway_public_network_name)
portal_public_network = slice.get_network(name=portal_public_network_name)
print (gateway_public_network)
print (portal_public_network)
gateway_node = slice.get_node(name=gateway_node_name)
portal_node = slice.get_node(name=portal_node_name)
service_node = slice.get_node(name=service_node_name)
gateway_node_public_iface = gateway_node.get_interface(network_name=gateway_public_network_name)
portal_node_public_iface = portal_node.get_interface(network_name=portal_public_network_name)

portal_public_addr = portal_public_network.get_public_ips()[0]
print (portal_public_addr)
gateway_public_addr = gateway_public_network.get_public_ips()[0]
print (gateway_public_addr)
gateway_node_public_iface.ip_addr_add(addr=gateway_public_addr, subnet=gateway_public_network.get_subnet())
portal_node_public_iface.ip_addr_add(addr=portal_public_addr, subnet=portal_public_network.get_subnet())


In [ ]:
from ipaddress import IPv4Network

PUBLIC_TABLE = 102
NETPLAN_FILE = '/etc/netplan/60-fabnet-public.yaml'
# Used only on VMs without netplan (e.g. Rocky), and removed from netplan VMs
ROUTE_SCRIPT = '/usr/local/sbin/fabnet-public-routes.sh'
ROUTE_SERVICE = 'fabnet-public-routes'
NETWORKD_DROPIN = '/etc/systemd/networkd.conf.d/keep-fabnet-routes.conf'


def uses_netplan(node):
    out = node.execute('command -v netplan >/dev/null && systemctl is-active --quiet systemd-networkd && echo yes',
                       quiet=True)[0]
    return out.strip() == 'yes'


def netplan_config(pub_iface, public_addr, prefix, pub_gw, extra_routes, fabnetv6):
    """Netplan YAML: public IP, a default route in table 102, the rule
    'from public IP -> table 102', extra /32 routes, and optionally the FABnetv6 route."""
    extra = ''.join(f"""
        - to: {dest}
          via: {pub_gw}
          from: {public_addr}""" for dest in extra_routes)
    yaml = f"""network:
  version: 2
  ethernets:
    {pub_iface}:
      addresses: [{public_addr}/{prefix}]
      routes:
        - to: default
          via: {pub_gw}
          table: {PUBLIC_TABLE}{extra}
      routing-policy:
        - from: {public_addr}
          table: {PUBLIC_TABLE}
          priority: 100"""
    if fabnetv6:
        v6_iface, v6_addr, v6_prefix, v6_gw = fabnetv6
        yaml += f"""
    {v6_iface}:
      addresses: [{v6_addr}/{v6_prefix}]
      routes:
        - to: 2602:fcfb::/40
          via: {v6_gw}"""
    return yaml


def configure_with_script(node, pub_iface, public_addr, prefix, pub_gw, extra_routes):
    """No netplan: a script re-applied at boot by a systemd service."""
    lines = [
        '#!/bin/bash',
        f'# Written by the slice setup notebook. Re-applied at boot by {ROUTE_SERVICE}.service',
        'set -e',
        f'ip link set dev {pub_iface} up',
        f'ip addr replace {public_addr}/{prefix} dev {pub_iface}',
        f'ip route replace default via {pub_gw} dev {pub_iface} table {PUBLIC_TABLE}',
        f'while ip rule del from {public_addr} table {PUBLIC_TABLE} 2>/dev/null; do :; done',
        f'ip rule add from {public_addr} table {PUBLIC_TABLE} priority 100',
    ] + [f'ip route replace {dest} via {pub_gw} dev {pub_iface} src {public_addr}' for dest in extra_routes]
    script = '\n'.join(lines)
    unit = f"""[Unit]
Description=FABNetv4Ext public IP and routes
After=network-online.target NetworkManager.service
Wants=network-online.target

[Service]
Type=oneshot
ExecStart={ROUTE_SCRIPT}
RemainAfterExit=yes

[Install]
WantedBy=multi-user.target"""
    node.execute(f"sudo tee {ROUTE_SCRIPT} > /dev/null <<'EOF'\n{script}\nEOF\n"
                 f"sudo chmod 755 {ROUTE_SCRIPT}", quiet=True)
    node.execute(f"sudo tee /etc/systemd/system/{ROUTE_SERVICE}.service > /dev/null <<'EOF'\n{unit}\nEOF\n"
                 f"sudo systemctl daemon-reload && sudo systemctl enable --quiet {ROUTE_SERVICE} "
                 f"&& sudo systemctl restart {ROUTE_SERVICE}", quiet=True)
    node.execute(f'systemctl is-active {ROUTE_SERVICE} || '
                 f'sudo journalctl -u {ROUTE_SERVICE} -b --no-pager | tail -20')


def configure_public_routes(node, public_iface, public_network, public_addr, extra_routes=(), fabnetv6=None):
    """Traffic from/to the public IP uses FABNetv4Ext; the main default route is not changed.
    extra_routes: destinations (e.g. 'x.x.x.x/32') sent out the public interface from the public IP.
    fabnetv6: optional (iface name, address, prefix length, gateway) for the FABnetv6 interface."""
    pub_iface = public_iface.get_device_name()
    pub_gw = str(public_network.get_gateway())
    prefix = IPv4Network(str(public_network.get_subnet())).prefixlen

    if uses_netplan(node):
        how = 'netplan'
        yaml = netplan_config(pub_iface, public_addr, prefix, pub_gw, extra_routes, fabnetv6)
        node.execute(f"sudo tee {NETPLAN_FILE} > /dev/null <<'EOF'\n{yaml}\nEOF\n"
                     f"sudo chmod 600 {NETPLAN_FILE}", quiet=True)
        # Show netplan errors instead of hiding them
        node.execute('sudo netplan generate && sudo netplan apply 2>&1 | grep -v "world-inaccessible" || true')
        # Remove the old boot-script setup if an earlier version of this cell installed it
        node.execute(f'sudo systemctl disable --now {ROUTE_SERVICE} 2>/dev/null; '
                     f'sudo rm -f /etc/systemd/system/{ROUTE_SERVICE}.service {ROUTE_SCRIPT} {NETWORKD_DROPIN}; '
                     f'sudo systemctl daemon-reload', quiet=True)
    else:
        how = 'boot script'
        configure_with_script(node, pub_iface, public_addr, prefix, pub_gw, extra_routes)

    print(f"\n===== {node.get_name()}: {public_addr}/{prefix} on {pub_iface} via {pub_gw} ({how}) =====")
    node.execute(f'ip -br addr show {pub_iface}; ip rule | grep {PUBLIC_TABLE}; '
                 f'ip route list table {PUBLIC_TABLE}')


configure_public_routes(gateway_node, gateway_node_public_iface, gateway_public_network, gateway_public_addr)
# Optional: also keep the gateway's FABnetv6 address and route in netplan, so they survive too.
# Replace the call above with this, using your notebook's FABnetv6 variables:
# configure_public_routes(gateway_node, gateway_node_public_iface, gateway_public_network, gateway_public_addr,
#                         fabnetv6=(gateway_v6_iface.get_device_name(), gateway_v6_addr,
#                                   gateway_v6_network.get_subnet().prefixlen, gateway_v6_network.get_gateway()))

# Portal -> gateway must leave from the portal's public IP, so the gateway allowlist sees it
configure_public_routes(portal_node, portal_node_public_iface, portal_public_network, portal_public_addr,
                        extra_routes=[f'{gateway_public_addr}/32'])

# Check the routes survive a networkd reconfigure (this is what deleted them before)
print('\nAfter a networkd reload:')
for node in (gateway_node, portal_node):
    node.execute('sudo networkctl reload 2>/dev/null; sleep 3', quiet=True)
gateway_node.execute(f'ip rule | grep -q "lookup {PUBLIC_TABLE}" && echo "gateway: rule kept" || echo "gateway: rule MISSING"')
print('Portal -> gateway route (want: dev <public iface> src <portal public IP>):')
portal_node.execute(f'ip route get {gateway_public_addr}')

print(f'\nPut this in gateway.env:  GATEWAY_ALLOWED_SOURCES={portal_public_addr}/32')

In [ ]:
# mgmt_interfacename = node1.get_management_os_interface()
# expt_interfacename = node1_public_iface.get_device_name()
# network2_gateway = str(network2.get_gateway())
# print (network2_gateway)
# # Add route to external network
# stdout, stderr = node1.execute(f'sudo echo -n "1 {mgmt_interfacename}\n" | sudo tee -a /etc/iproute2/rt_tables')
# stdout, stderr = node1.execute(f'sudo echo -n "2 {expt_interfacename}\n" | sudo tee -a /etc/iproute2/rt_tables')
# stdout, stderr = node1.execute(f'sudo ip route add 0.0.0.0/0 via 10.30.6.11 dev {mgmt_interfacename} table {mgmt_interfacename}')
# stdout, stderr = node1.execute(f'sudo ip route add 0.0.0.0/0 via {network2_gateway} dev {expt_interfacename} table {expt_interfacename}')
# stdout, stderr = node1.execute(f'sudo ip rule add from {node1_public_addr}/32 table {expt_interfacename}')
# stdout, stderr = node1.execute(f'sudo ip rule add to {node1_public_addr}/32 table {expt_interfacename}')
# stdout, stderr = node1.execute(f'sudo ip route add default via {network2_gateway} dev {expt_interfacename}')
# stdout, stderr = node1.execute(f'ip addr show {expt_interfacename}')    
# stdout, stderr = node1.execute(f'ip route list table {mgmt_interfacename}')
# stdout, stderr = node1.execute(f'ip route list table {expt_interfacename}')



In [ ]:
portal_node.execute("sudo DEBIAN_FRONTEND=noninteractive apt update && sudo DEBIAN_FRONTEND=noninteractive apt upgrade -y && sudo DEBIAN_FRONTEND=noninteractive apt install -y python3 python3-pip python3-venv nginx sqlite3")
gateway_node.execute("sudo DEBIAN_FRONTEND=noninteractive apt update && sudo DEBIAN_FRONTEND=noninteractive apt upgrade -y && sudo DEBIAN_FRONTEND=noninteractive apt install -y python3 python3-pip python3-venv nginx sqlite3")
service_node.execute("sudo dnf install -y iputils traceroute iperf3 python3-pip")
service_node.execute("pip3 install --user flask")

In [ ]:
slice_name = 'service_marketplace'
slice = fablib.get_slice(name = slice_name)
portal_node = slice.get_node(name=portal_node_name)
portal_node.upload_directory("fabric_service_marketplace", ".")
gateway_node = slice.get_node(name=gateway_node_name)
gateway_node.upload_directory("fabric_service_marketplace", ".")